# Model Training

In [ ]:
import joblib

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import (
    RandomForestClassifier,
    GradientBoostingClassifier,
    ExtraTreesClassifier,
    HistGradientBoostingClassifier
)
from sklearn.metrics import accuracy_score, classification_report
from sklearn.model_selection import train_test_split



Load data

In [ ]:
df_matches_with_features = pd.read_csv("model/matches_with_features.csv")

Split features and labels

In [ ]:
feature_cols = [
    "total_goals_team1",
    "num_players_with_goals_team1",
    "max_goals_by_single_player_team1",
    "total_matches_team1",
    "matches_with_goals_team1",
    "matches_without_goals_team1",
    "avg_goals_per_match_team1",
    "total_goals_team2",
    "num_players_with_goals_team2",
    "max_goals_by_single_player_team2",
    "total_matches_team2",
    "matches_with_goals_team2",
    "matches_without_goals_team2",
    "avg_goals_per_match_team2",
    "goals_diff",
    "matches_diff",
    "avg_goals_diff",
    "num_players_with_goals_diff",
    "max_goals_diff"
]

X = df_matches_with_features[feature_cols]
y = df_matches_with_features["win"]
model_choice = input(
    "Choose model:\n"
    "* logreg  - Logistic Regression\n"
    "* rf      - Random Forest Classifier\n"
    "* extratrees      - ExtraTrees Classifier\n"
    "* gb      - Gradient Boosting Classifier\n"
    "* histgb     - Histogram Gradient Boosting Classifier\n"
).strip().lower()

Split to training and testing dataset

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

Training with model of user's choice

In [ ]:

models = {
    "logreg": LogisticRegression(max_iter=1000, random_state=42),
    "rf": RandomForestClassifier(n_estimators=200, random_state=42),
    "extratrees": ExtraTreesClassifier(n_estimators=200, random_state=42),
    "gb": GradientBoostingClassifier(n_estimators=200, random_state=42),
    "histgb": HistGradientBoostingClassifier(max_iter=200, learning_rate=0.05, random_state=42)
}

if model_choice not in models:
    raise ValueError(f"Invalid model. Choose from: {list(models.keys())}")

print(f"Selected model: {model_choice}")
clf = models[model_choice]
clf.fit(X_train, y_train)

Output model, feature columns and metadata

In [ ]:
joblib.dump(clf, f"model/{model_choice}/worldcup_model.pkl")
joblib.dump(feature_cols, f"model/{model_choice}/feature_columns.pkl")
model_info = {
    "model": "RandomForestClassifier",
    "trained_on": "Worldcup data 2002 - 2022",
    "features": feature_cols
}

joblib.dump(model_info, f"model/{model_choice}/model_metadata.pkl")

Evaluation

In [ ]:
if model_choice == "rf":
    y_pred = clf.predict(X_test)
    accuracy = accuracy_score(y_test, y_pred)
    print("Accuracy:", accuracy)
    print(classification_report(y_test, y_pred))

    feat_imp = pd.DataFrame({
        "feature": X_train.columns,
        "importance": clf.feature_importances_
    }).sort_values(by="importance", ascending=False)

    print(feat_imp.head(10))

Visualization

In [ ]:
if model_choice == "rf":
    top_features = feat_imp.sort_values(by="importance", ascending=False).head(10)

    plt.figure(figsize=(10,6))
    sns.barplot(
        x="importance",
        y="feature",
        data=top_features,
        palette="viridis"
    )
    plt.title("Top 10 Features influencing Match Outcome")
    plt.xlabel("Feature Importance")
    plt.ylabel("Feature")
    plt.tight_layout()
    plt.show()

In [ ]:
y_prob = clf.predict_proba(X_test)[:,1]

plt.figure(figsize=(8,5))
sns.histplot(y_prob, bins=20, kde=True, color="skyblue")
plt.title("Predicted Probability of Team1 Winning")
plt.xlabel("Predicted Probability")
plt.ylabel("Number of Matches")
plt.tight_layout()
plt.show()